# 🏥 Sistema de Gestion Hospitalaria (SGH)

**Programacion Orientada a Objetos**

---

### Instrucciones
1. Ejecutar todas las celdas (Ctrl+F9)
2. La aplicacion se mostrara al final
3. Los datos se guardan en Google Drive

In [1]:
# Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')
RUTA_BASE = '/content/drive/MyDrive/SGH/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)
RUTAS = {'pacientes': RUTA_BASE + 'pacientes.csv', 'medicos': RUTA_BASE + 'medicos.csv', 'especialidades': RUTA_BASE + 'especialidades.csv', 'citas': RUTA_BASE + 'citas.csv', 'consultas': RUTA_BASE + 'consultas.csv', 'tratamientos': RUTA_BASE + 'tratamientos.csv', 'medicamentos': RUTA_BASE + 'medicamentos.csv'}
print('Drive OK')

Mounted at /content/drive
Drive OK


In [2]:
# Dependencias
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
print('Widgets OK')

Widgets OK


In [8]:

# === MODELOS ===
from enum import Enum
from datetime import datetime
from collections import Counter

class Paciente:
    def __init__(self, num_documento, nombre, fecha_nacimiento, tipo_sangre, eps, regimen, antecedentes='NINGUNO'):
        self.num_documento, self.nombre, self.fecha_nacimiento = num_documento, nombre, fecha_nacimiento
        self.tipo_sangre, self.eps, self.regimen, self.antecedentes = tipo_sangre, eps, regimen, antecedentes
    def to_linea(self): return f"{self.num_documento}|{self.nombre}|{self.fecha_nacimiento}|{self.tipo_sangre}|{self.eps}|{self.regimen}|{self.antecedentes}"
    @staticmethod
    def from_linea(linea):
        c = linea.strip().split('|')
        return Paciente(c[0], c[1], c[2], c[3], c[4], c[5], c[6] if len(c) > 6 else 'NINGUNO')

class Medico:
    def __init__(self, num_registro, nombre, especialidad, consultorio, horario):
        self.num_registro, self.nombre, self.especialidad = num_registro, nombre, especialidad
        self.consultorio, self.horario = consultorio, horario
    def to_linea(self): return f"{self.num_registro}|{self.nombre}|{self.especialidad}|{self.consultorio}|{self.horario}"
    @staticmethod
    def from_linea(l):
        linea = l.strip()
        if not linea or linea.lower() in ['num_registro,nombre,especialidad,consultorio,horario']: return None
        c = linea.split('|')
        if len(c) < 5: return None
        return Medico(c[0], c[1], c[2], c[3], c[4])

class Especialidad:
    def __init__(self, codigo, nombre, descripcion): self.codigo, self.nombre, self.descripcion = codigo, nombre, descripcion
    def to_linea(self): return f"{self.codigo},{self.nombre},{self.descripcion}"
    @staticmethod
    def from_linea(l):
        linea = l.strip()
        if not linea or linea.lower() in ['codigo,nombre,descripcion', 'codigo,nombre,desc']: return None
        c = linea.split(',')
        if len(c) < 3: return None
        return Especialidad(c[0], c[1], c[2])

class Cita:
    def __init__(self, codigo, num_paciente, num_medico, fecha, hora, motivo, estado='PROGRAMADA'):
        self.codigo, self.num_paciente, self.num_medico = codigo, num_paciente, num_medico
        self.fecha, self.hora, self.motivo, self.estado = fecha, hora, motivo, estado
    def to_linea(self): return f"{self.codigo}|{self.num_paciente}|{self.num_medico}|{self.fecha}|{self.hora}|{self.motivo}|{self.estado}"
    @staticmethod
    def from_linea(l): c = l.strip().split('|'); return Cita(c[0], c[1], c[2], c[3], c[4], c[5], c[6] if len(c) > 6 else 'PROGRAMADA')

class Diagnostico:
    def __init__(self, codigo_cie10, descripcion): self.codigo_cie10, self.descripcion = codigo_cie10, descripcion
    def __str__(self): return f"[{self.codigo_cie10}] {self.descripcion}"

class ConsultaMedica:
    def __init__(self, codigo, codigo_cita, fecha, sintomas, presion, temperatura, frec_cardiaca, diagnostico, observaciones):
        self.codigo, self.codigo_cita, self.fecha = codigo, codigo_cita, fecha
        self.sintomas, self.presion, self.temperatura = sintomas, presion, temperatura
        self.frec_cardiaca, self.diagnostico, self.observaciones = frec_cardiaca, diagnostico, observaciones
    def to_linea(self): return f"{self.codigo}|{self.codigo_cita}|{self.fecha}|{self.sintomas}|{self.presion}|{self.temperatura}|{self.frec_cardiaca}|{self.diagnostico.codigo_cie10}|{self.diagnostico.descripcion}|{self.observaciones}"
    @staticmethod
    def from_linea(l):
        c = l.strip().split('|')
        return ConsultaMedica(c[0], c[1], c[2], c[3], c[4], c[5], c[6], Diagnostico(c[7], c[8]), c[9])

class Tratamiento:
    def __init__(self, codigo, codigo_consulta, fecha_inicio, duracion_dias):
        self.codigo, self.codigo_consulta, self.fecha_inicio = codigo, codigo_consulta, fecha_inicio
        self.duracion_dias = int(duracion_dias)
    def to_linea(self): return f"{self.codigo}|{self.codigo_consulta}|{self.fecha_inicio}|{self.duracion_dias}"
    @staticmethod
    def from_linea(l): c = l.strip().split('|'); return Tratamiento(c[0], c[1], c[2], c[3])

class Medicamento:
    def __init__(self, cod_tratamiento, nombre, dosis, frecuencia, duracion_dias):
        self.cod_tratamiento, self.nombre = cod_tratamiento, nombre
        self.dosis, self.frecuencia, self.duracion_dias = dosis, frecuencia, int(duracion_dias)
    def to_linea(self): return f"{self.cod_tratamiento}|{self.nombre}|{self.dosis}|{self.frecuencia}|{self.duracion_dias}"
    @staticmethod
    def from_linea(l): c = l.strip().split('|'); return Medicamento(c[0], c[1], c[2], c[3], c[4])

# === REPOSITORIOS ===
class ArchivoUtil:
    @staticmethod
    def leer_lineas(ruta):
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                lines = [l.strip() for l in f if l.strip()]
                return lines[1:] if lines and 'documento' in lines[0].lower() else lines
        except: return []

    @staticmethod
    def escribir_linea(ruta, linea):
        import os
        es_nuevo = not os.path.exists(ruta) or os.path.getsize(ruta) == 0
        with open(ruta, 'a', encoding='utf-8') as f:
            if es_nuevo:
                headers = {'pacientes': 'num_documento,nombre,fecha_nacimiento,tipo_sangre,eps,regimen,antecedentes', 'medicos': 'num_registro,nombre,especialidad,consultorio,horario', 'especialidades': 'codigo,nombre,descripcion', 'citas': 'codigo,num_paciente,num_medico,fecha,hora,motivo,estado', 'consultas': 'codigo,codigo_cita,fecha,sintomas,presion,temperatura,frec_cardiaca,codigo_cie10,descripcion_diagnostico,observaciones', 'tratamientos': 'codigo,codigo_consulta,fecha_inicio,duracion_dias', 'medicamentos': 'cod_tratamiento,nombre,dosis,frecuencia,duracion_dias'}
                for k, h in headers.items():
                    if k in ruta: f.write(h + '\n'); break
            f.write(linea + '\n')

    @staticmethod
    def reescribir(ruta, lineas):
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lineas) + '\n' if lineas else '')

    @staticmethod
    def generar_codigo(prefijo):
        return f"{prefijo}-{datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]}"

class Repositorio:
    def __init__(self, ruta, modelo): self.ruta, self.modelo = ruta, modelo
    def guardar(self, obj): ArchivoUtil.escribir_linea(self.ruta, obj.to_linea())
    def find_all(self): return [x for x in [self.modelo.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)] if x is not None]
    def reescribir(self, nuevos): ArchivoUtil.reescribir(self.ruta, [n.to_linea() for n in nuevos])

# === SERVICIOS ===
class Servicio:
    def __init__(self, repo): self.repo = repo

class PacienteService(Servicio):
    def registrar(self, num_documento, nombre, fecha_nacimiento, tipo_sangre, eps, regimen, antecedentes='NINGUNO'):
        if self.repo.find_all() and any(p.num_documento == num_documento for p in self.repo.find_all()): return False, 'Ya existe'
        self.repo.guardar(Paciente(num_documento, nombre, fecha_nacimiento, tipo_sangre, eps, regimen, antecedentes))
        return True, self.repo.find_all()[-1]
    def buscar(self, doc): return next((p for p in self.repo.find_all() if p.num_documento == doc), None)
    def actualizar(self, num_documento, eps=None, regimen=None, antecedentes=None):
        pacientes = self.repo.find_all()
        for p in pacientes:
            if p.num_documento == num_documento:
                if eps: p.eps = eps
                if regimen: p.regimen = regimen
                if antecedentes: p.antecedentes = antecedentes
                self.repo.reescribir(pacientes)
                return True, p
        return False, 'No encontrado'
    def listar_todos(self): return self.repo.find_all()

class MedicoService(Servicio):
    def registrar(self, num_registro, nombre, especialidad, consultorio, horario):
        if self.repo.find_all() and any(m.num_registro == num_registro for m in self.repo.find_all()): return False, 'Ya existe'
        self.repo.guardar(Medico(num_registro, nombre, especialidad, consultorio, horario))
        return True, self.repo.find_all()[-1]
    def buscar(self, reg): return next((m for m in self.repo.find_all() if m.num_registro == reg), None)
    def listar_todos(self): return self.repo.find_all()
    def listar_por_especialidad(self, esp): return [m for m in self.repo.find_all() if m.especialidad.upper() == esp.upper()]

class EspecialidadService(Servicio):
    def registrar(self, codigo, nombre, descripcion):
        if self.repo.find_all() and any(e.codigo == codigo for e in self.repo.find_all()): return False, 'Ya existe'
        self.repo.guardar(Especialidad(codigo, nombre, descripcion))
        return True, self.repo.find_all()[-1]
    def listar_todas(self): return self.repo.find_all()

class CitaService(Servicio):
    def __init__(self, repo, pac_repo, med_repo): super().__init__(repo); self.pac_repo, self.med_repo = pac_repo, med_repo
    def programar(self, num_paciente, num_medico, fecha, hora, motivo):
        if not self.pac_repo.find_all() or not any(p.num_documento == num_paciente for p in self.pac_repo.find_all()): return False, 'Paciente no existe'
        if not self.med_repo.find_all() or not any(m.num_registro == num_medico for m in self.med_repo.find_all()): return False, 'Medico no existe'
        cita = Cita(ArchivoUtil.generar_codigo('CIT'), num_paciente, num_medico, fecha, hora, motivo)
        self.repo.guardar(cita)
        return True, cita
    def buscar(self, cod): return next((c for c in self.repo.find_all() if c.codigo == cod), None)
    def cancelar(self, cod):
        citas = self.repo.find_all()
        for c in citas:
            if c.codigo == cod:
                c.estado = 'CANCELADA'
                self.repo.reescribir(citas)
                return True, 'Cita cancelada'
        return False, 'No encontrada'
    def listar_todas(self): return self.repo.find_all()

# === UI ===
CSS = """<style>.sgh-root{font-family:sans-serif}.sgh-header{background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);color:#e8e8e8;padding:18px;border-radius:10px 10px 0 0}.sgh-card{background:#252541;border:1px solid #3d3d5c;border-radius:8px;padding:20px;margin:10px 0}.sgh-card h3{color:#00d9ff;margin-bottom:14px;border-bottom:2px solid #3d3d5c;padding-bottom:8px}.sgh-ok{color:#00d9a5;background:#1a3a2e;border:1px solid #00d9a5;padding:8px;border-radius:6px;margin:8px 0}.sgh-err{color:#ff6b6b;background:#3d1a1a;border:1px solid #ff6b6b;padding:8px;border-radius:6px;margin:8px 0}.sgh-info{color:#00d9ff;background:#1a2a3d;border:1px solid #00d9ff;padding:8px;border-radius:6px;margin:8px 0}.sgh-table{width:100%;border-collapse:collapse;color:#d0d0d0}.sgh-table th{background:#16213e;color:#00d9ff;padding:8px;text-align:left}.sgh-table td{padding:7px;border-bottom:1px solid #3d3d5c}.sgh-badge{padding:2px 9px;border-radius:12px;font-size:.75rem}.badge-prog{background:#4a3d00;color:#ffd700}.badge-aten{background:#003d2a;color:#00d9a5}.badge-canc{background:#3d1a1a;color:#ff6b6b}.sgh-nav{display:flex;gap:8px;background:#1a1a2e;padding:12px;border:1px solid #3d3d5c;border-top:none;border-radius:0 0 10px 10px;flex-wrap:wrap}.sgh-stat-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(160px,1fr));gap:12px;margin-top:12px}.sgh-stat{background:#2d2d4a;border-radius:8px;padding:14px;text-align:center}.sgh-stat .num{font-size:1.8rem;font-weight:700;color:#00d9ff}.sgh-stat .lbl{font-size:.75rem;color:#8888aa}</style>"""

def badge(e): cl = {'PROGRAMADA':'badge-prog','ATENDIDA':'badge-aten','CANCELADA':'badge-canc'}.get(e,''); return f'<span class="sgh-badge {cl}">{e}</span>'
def alert(m, t='ok'): display(HTML(f'<div class="sgh-{t}">{m}</div>'))
def txt(l, p='', w='260px'): return widgets.Text(description=l+':', placeholder=p, layout=widgets.Layout(width=w), style={'description_width':'140px'})
def dd(l, o, w='300px'): return widgets.Dropdown(description=l+':', options=o, layout=widgets.Layout(width=w), style={'description_width':'140px'})
def btn(l, c='#0f4c75'): return widgets.Button(description=l, layout=widgets.Layout(width='auto', height='34px'), style=widgets.ButtonStyle(button_color=c, font_weight='600'))
def show_css(): display(HTML(CSS))

def validate_time_input(time_str):
    try:
        if not time_str.strip():
            return None
        return datetime.strptime(time_str, '%H:%M').time()
    except ValueError:
        return None

def panel_pacientes(svc):
    show_css(); display(HTML('<div class="sgh-card"><h3>Gestion de Pacientes</h3><p style="color:#888">Administra los datos de los pacientes.</p></div>'))
    tabs = widgets.Tab()
    f_doc = widgets.IntText(description='N Documento:', placeholder='ej. 1098765432', layout=widgets.Layout(width='260px'), style={'description_width':'140px'})
    f_nom, f_fec = txt('Nombre completo'), widgets.DatePicker(description='Fecha nacimiento:', style={'description_width':'140px'}, layout=widgets.Layout(width='260px'))
    f_san = dd('Tipo sangre', ['Seleccionar...','A+','A-','B+','B-','AB+','AB-','O+','O-'], '280px')
    f_eps, f_reg = txt('EPS'), dd('Regimen', ['Seleccionar...','CONTRIBUTIVO','SUBSIDIADO'], '280px')
    f_ant = txt('Antecedentes', 'Tos', '340px')
    b_reg, out_r = btn('Registrar', '#1a7f4b'), widgets.Output()
    def on_reg(_):
        with out_r:
            clear_output()
            fecha_str = f_fec.value.strftime('%Y-%m-%d') if f_fec.value else ''
            if not f_doc.value or f_doc.value == 0: alert('Error: Ingrese documento', 'err'); return
            if not fecha_str: alert('Error: Seleccione fecha', 'err'); return
            if f_san.value == 'Seleccionar...': alert('Error: Seleccione tipo sangre', 'err'); return
            if f_reg.value == 'Seleccionar...': alert('Error: Seleccione regimen', 'err'); return
            ok, res = svc.registrar(str(f_doc.value), f_nom.value.strip(), fecha_str, f_san.value, f_eps.value.strip(), f_reg.value, f_ant.value.strip() or 'NINGUNO')
            alert(f'Paciente {res.nombre} registrado' if ok else f'Error: {res}', 'ok' if ok else 'err')
            if ok:
                f_doc.value = 0; f_nom.value = ''; f_fec.value = None; f_san.value = 'Seleccionar...'; f_eps.value = ''; f_reg.value = 'Seleccionar...'; f_ant.value = ''
    b_reg.on_click(on_reg)
    tab0 = widgets.VBox([f_doc, f_nom, f_fec, f_san, f_eps, f_reg, f_ant, b_reg, out_r], layout=widgets.Layout(padding='12px'))
    b_doc2, b_bus, b_limp = widgets.IntText(description='N Documento:', layout=widgets.Layout(width='260px'), style={'description_width':'140px'}), btn('Buscar'), btn('Limpiar')
    out_b = widgets.Output()
    def on_bus(_):
        with out_b:
            clear_output()
            if not b_doc2.value or b_doc2.value == 0: alert('Error: Ingrese documento', 'err'); return
            p = svc.buscar(str(b_doc2.value))
            if p: display(HTML(f'<table class="sgh-table"><tr><th>Campo</th><th>Valor</th></tr><tr><td>Documento</td><td>{p.num_documento}</td></tr><tr><td>Nombre</td><td>{p.nombre}</td></tr><tr><td>Fecha</td><td>{'/'.join(p.fecha_nacimiento.split('-')[::-1])}</td></tr><tr><td>Tipo</td><td>{p.tipo_sangre}</td></tr><tr><td>EPS</td><td>{p.eps}</td></tr><tr><td>Regimen</td><td>{p.regimen}</td></tr></table>')); b_doc2.value = 0
            else: alert('Paciente no encontrado', 'err')
    b_bus.on_click(on_bus)
    def on_limp(_): out_b.clear_output(); b_doc2.value = 0; alert('Campos limpiados', 'info')
    b_limp.on_click(on_limp)
    tab1 = widgets.VBox([b_doc2, widgets.HBox([b_bus, b_limp]), out_b], layout=widgets.Layout(padding='12px'))
    b_list, b_ref_l = btn('Listar todos'), btn('Refrescar', '#475569')
    out_l = widgets.Output()

    def on_list(_):
        with out_l:
            clear_output()
            pacs = svc.listar_todos()
            if not pacs: alert('No hay pacientes', 'info'); return
            filas = ''.join(f'<tr><td>{p.num_documento}</td><td>{p.nombre}</td><td>{'/'.join(p.fecha_nacimiento.split('-')[::-1]) if p.fecha_nacimiento else p.fecha_nacimiento}</td><td>{p.eps}</td><td>{p.regimen}</td></tr>' for p in pacs)
            display(HTML(f'<table class="sgh-table"><tr><th>Documento</th><th>Nombre</th><th>Fecha</th><th>EPS</th><th>Regimen</th></tr>{filas}</table>'))

    def on_ref_list(_):
        on_list(None)

    b_list.on_click(on_list)
    b_ref_l.on_click(on_ref_list)
    tab2 = widgets.VBox([b_list, b_ref_l, out_l], layout=widgets.Layout(padding='12px'))
    def get_pacientes():
        return [(f'{p.num_documento} - {p.nombre}', p.num_documento) for p in svc.listar_todos()] if svc.listar_todos() else [('Sin pacientes', '')]

    u_sel = dd('Paciente:', get_pacientes())
    u_eps, u_ant = txt('Nueva EPS'), txt('Antecedentes', 'Tos', '340px')
    u_reg = dd('Nuevo Regimen', ['Sin cambios', 'CONTRIBUTIVO','SUBSIDIADO'], '280px')
    b_ref = btn('Refrescar', '#475569')
    b_upd, out_u = btn('Actualizar', '#b45309'), widgets.Output()

    def on_ref(_):
        u_sel.options = get_pacientes()

    b_ref.on_click(on_ref)

    def on_upd(_):
        with out_u:
            clear_output()
            doc = u_sel.value
            if not doc or doc == 'Sin pacientes' or doc is None: alert('Seleccione paciente', 'err'); return
            regimen = None if u_reg.value == 'Sin cambios' else u_reg.value
            ok, res = svc.actualizar(doc, eps=u_eps.value.strip() or None, regimen=regimen, antecedentes=u_ant.value.strip() or None)
            if ok: alert(f'Paciente {res.nombre} actualizado', 'ok'); u_sel.options = get_pacientes(); u_eps.value = ''; u_reg.value = 'Sin cambios'; u_ant.value = ''
            else: alert(f'Error: {res}', 'err')
    b_upd.on_click(on_upd)
    tab3 = widgets.VBox([u_sel, b_ref, u_eps, u_reg, u_ant, b_upd, out_u], layout=widgets.Layout(padding='12px'))
    tabs.children = [tab0, tab1, tab2, tab3]
    for i, t in enumerate(['Registrar','Consultar','Listar','Actualizar']): tabs.set_title(i, t)
    display(tabs)

def panel_medicos(svc_med, svc_esp):
    show_css(); display(HTML('<div class="sgh-card"><h3>Gestion de Medicos</h3><p style="color:#888">Administra especialidades y medicos del hospital.</p></div>'))
    tabs = widgets.Tab()

    # TAB: Especialidad
    e_cod, e_nom, e_desc = widgets.IntText(description='Cedula:', placeholder='ej. 101', layout=widgets.Layout(width='200px'), style={'description_width':'80px'}), txt('Nombre Especialidad', 'Medicina General'), txt('Descripcion', 'ej. Cardiologia, Pediatra', '340px')
    b_esp, out_e = btn('Guardar Especialidad', '#1a7f4b'), widgets.Output()
    def on_esp(_):
        with out_e:
            clear_output()
            if not str(e_cod.value) or not e_nom.value.strip(): alert('Ingrese codigo y nombre', 'err'); return
            ok, res = svc_esp.registrar(str(e_cod.value).upper(), e_nom.value.strip(), e_desc.value.strip())
            if ok:
                alert(f'Especialidad {res.nombre} registrada', 'ok')
                e_cod.value = 0; e_nom.value = ''; e_desc.value = ''; 
                listar_esp()
                # Actualizar automaticamente el dropdown de medicos
                m_esp.options = get_especialidades()
                if get_especialidades() and get_especialidades()[0][1]:
                    m_esp.value = get_especialidades()[0][1]
            else:
                alert(f'Error: {res}', 'err')
    b_esp.on_click(on_esp)
    def listar_esp():
        with out_e:
            clear_output()
            esp_list = svc_esp.listar_todas()
            if esp_list:
                filas = ''.join(f'<tr><td>{e.codigo}</td><td>{e.nombre}</td><td>{e.descripcion}</td></tr>' for e in esp_list)
                display(HTML('<table class="sgh-table"><thead><tr><th>Codigo</th><th>Nombre Especialidad</th><th>Descripcion</th></tr></thead><tbody>' + filas + '</tbody></table>'))
            else:
                display(HTML('<p style="color:#888">No hay especialidades registradas</p>'))
    out_e = widgets.Output()
    listar_esp()
    tab0 = widgets.VBox([e_cod, e_nom, e_desc, b_esp, out_e], layout=widgets.Layout(padding='12px'))

    # TAB: Medico - Dropdown dinamico de especialidades
    # Funcion para obtener lista de especialidades (evita duplicados)
    def get_especialidades():
        especialidades = svc_esp.listar_todas()
        if not especialidades:
            return [('Sin especialidades', '')]
        # Eliminar duplicados y crear lista
        seen = set()
        opciones = []
        for e in especialidades:
            if e.nombre not in seen:
                seen.add(e.nombre)
                opciones.append((e.nombre, e.nombre))
        return opciones
    
    # Crear dropdown con diseño profesional
    m_esp = widgets.Dropdown(
        options=get_especialidades(),
        value='',
        description='Especialidad:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='280px', height='35px')
    )
    
    # Campos del médico
    m_reg, m_nom = txt('Registro', 'MED-001'), txt('Nombre del Especialista')
    horas = ['07:00','07:30','08:00','08:30','09:00','09:30','10:00','10:30','11:00','11:30','12:00','12:30','13:00','13:30','14:00','14:30','15:00','15:30','16:00','16:30','17:00','17:30','18:00']
    m_con, m_hor_date, m_hor_time = txt('Consultorio', 'C-12'), widgets.DatePicker(description='Fecha:', style={'description_width':'80px'}, layout=widgets.Layout(width='180px')), widgets.Dropdown(options=['Seleccionar...']+horas, value='Seleccionar...', description='Hora:', style={'description_width':'50px'}, layout=widgets.Layout(width='160px'))
    
    # Botones
    b_ref_esp = btn('Refrescar', '#475569')
    b_med, out_m = btn('Registrar Medico', '#1a7f4b'), widgets.Output()

    # Funcion para actualizar el dropdown despues de guardar especialidad
    def actualizar_dropdown_especialidades():
        opciones = get_especialidades()
        m_esp.options = opciones
        # Seleccionar primer valor valido si existe
        if opciones and opciones[0][1]:
            m_esp.value = opciones[0][1]
        else:
            m_esp.value = ''

    # Evento: Refrescar lista de especialidades
    def on_ref_esp(_):
        actualizar_dropdown_especialidades()
        alert('Especialidades actualizadas', 'ok')
    b_ref_esp.on_click(on_ref_esp)

    # Evento: Registrar medico
    def on_med(_):
        with out_m:
            clear_output()
            # Validar registro y nombre
            if not m_reg.value.strip() or not m_nom.value.strip(): 
                alert('Ingrese registro y nombre del medico', 'err'); return
            
            # Validar especialidad seleccionada
            if not m_esp.value or m_esp.value == '' or m_esp.value == 'Seleccionar...':
                alert('Seleccione una especialidad', 'err'); return
                
            # Validar fecha
            fecha_str = m_hor_date.value.strftime('%Y-%m-%d') if m_hor_date.value else ''
            if not fecha_str: 
                alert('Error: Seleccione una fecha para el horario', 'err'); return

            # Validar hora
            hora_seleccionada = m_hor_time.value
            if hora_seleccionada == 'Seleccionar...': 
                alert('Error: Seleccione una hora', 'err'); return
            hora_str = hora_seleccionada

            # Crear horario completo y registrar
            horario_completo = f'{fecha_str} {hora_str}'
            ok, res = svc_med.registrar(m_reg.value.strip(), m_nom.value.strip(), m_esp.value, m_con.value.strip(), horario_completo)
            
            if ok:
                alert(f'Medico {res.nombre} registrado correctamente', 'ok')
                # Limpiar formulario
                m_reg.value = ''
                m_nom.value = ''
                m_con.value = ''
                m_hor_date.value = None
                m_hor_time.value = 'Seleccionar...'
                m_esp.options = [('Seleccionar...', '')] + get_especialidades()
                m_esp.value = ''
            else:
                alert(f'Error: {res}', 'err')
    b_med.on_click(on_med)
    tab1 = widgets.VBox([m_reg, m_nom, widgets.HBox([m_esp, b_ref_esp]), m_con, widgets.HBox([m_hor_date, m_hor_time]), b_med, out_m], layout=widgets.Layout(padding='12px'))
        # ===== PESTAÑA: LISTAR MÉDICOS =====
    b_list, b_refresh, out_l = btn('Listar', '#1a7f4b'), btn('Refrescar', '#475569'), widgets.Output()
    
    # Función para listar médicos (reutilizable)
    def listar_medicos():
        with out_l:
            clear_output()
            meds = svc_med.listar_todos()
            if meds:
                filas = ''.join(f'<tr><td>{m.num_registro}</td><td>{m.nombre}</td><td>{m.especialidad}</td><td>{m.consultorio}</td></tr>' for m in meds)
                display(HTML('<table class="sgh-table"><tr><th>Registro</th><th>Nombre</th><th>Especialidad</th><th>Consultorio</th></tr>' + filas + '</table>'))
            else:
                display(HTML('<p style="color:#888">No hay medicos registrados</p>'))
    
    def on_list(_): listar_medicos()
    b_list.on_click(on_list)
    
    def on_refresh(_):
        listar_medicos()
    b_refresh.on_click(on_refresh)
    
    tab2 = widgets.VBox([widgets.HBox([b_list, b_refresh]), out_l], layout=widgets.Layout(padding='12px'))
    tabs.children = [tab0, tab1, tab2]
    for i, t in enumerate(['Especialidad','Medico','Listar']): tabs.set_title(i, t)
    display(tabs)

def panel_citas(svc_cit, svc_pac, svc_med):
    show_css(); display(HTML('<div class="sgh-card"><h3>Gestion de Citas</h3><p style="color:#888">Programa y administra las citas medicas del hospital.</p></div>'))
    tabs = widgets.Tab()

    def get_pacientes(): return [(f'{p.nombre} - {p.num_documento}', p.num_documento) for p in svc_pac.listar_todos()] if svc_pac.listar_todos() else [('Sin pacientes','')]
    def get_medicos(): return [(f'{m.nombre} [{m.especialidad}]', m.num_registro) for m in svc_med.listar_todos()] if svc_med.listar_todos() else [('Sin medicos','')]

    c_pac, c_med = dd('Paciente', get_pacientes()), dd('Medico', get_medicos())
        # Lista de horas predefinidas (igual al modulo de medicos)
    horas = ['07:00','07:30','08:00','08:30','09:00','09:30','10:00','10:30','11:00','11:30','12:00','12:30','13:00','13:30','14:00','14:30','15:00','15:30','16:00','16:30','17:00','17:30','18:00']
    c_fec, c_hor = widgets.DatePicker(description='Fecha:', style={'description_width':'80px'}, layout=widgets.Layout(width='180px')), widgets.Dropdown(options=['Seleccionar...']+horas, value='Seleccionar...', description='Hora:', style={'description_width':'50px'}, layout=widgets.Layout(width='160px'))
    c_mot = txt('Motivo', 'ej. Control mensual', '340px')
    b_ref_opts = btn('Refrescar listas', '#475569')
    b_prog, out_p = btn('Programar', '#1a7f4b'), widgets.Output()

    def on_ref_opts(_):
        c_pac.options = get_pacientes()
        c_med.options = get_medicos()
    b_ref_opts.on_click(on_ref_opts)

    def on_prog(_):
        with out_p:
            clear_output()
            # Validar paciente
            if not c_pac.value or c_pac.value == 'Sin pacientes': 
                alert('Seleccione un paciente', 'err'); return
            # Validar medico
            if not c_med.value or c_med.value == 'Sin medicos': 
                alert('Seleccione un medico', 'err'); return
            # Validar fecha
            fecha_str = c_fec.value.strftime('%Y-%m-%d') if c_fec.value else ''
            if not fecha_str: 
                alert('Seleccione una fecha', 'err'); return
            # Validar hora (dropdown)
            hora_seleccionada = c_hor.value
            if hora_seleccionada == 'Seleccionar...': 
                alert('Seleccione una hora', 'err'); return
            hora_str = hora_seleccionada

            # Programar cita
            ok, res = svc_cit.programar(c_pac.value, c_med.value, fecha_str, hora_str, c_mot.value.strip())
            
            if ok:
                alert(f'Cita {res.codigo} programada correctamente', 'ok')
                # Limpiar formulario
                c_pac.options = get_pacientes()
                c_med.options = get_medicos()
                c_fec.value = None
                c_hor.value = 'Seleccionar...'
                c_mot.value = ''
            else:
                alert(f'Error: {res}', 'err')
    b_prog.on_click(on_prog)
    tab0 = widgets.VBox([widgets.HBox([c_pac, c_med, b_ref_opts]), widgets.HBox([c_fec, c_hor]), c_mot, b_prog, out_p], layout=widgets.Layout(padding='12px'))

        # ===== CONSULTA Y CANCELAR CITAS =====
    q_cod = txt('Codigo', 'CIT-...')
    b_bus, b_list_all, b_canc = btn('Buscar'), btn('Mostrar Todas', '#3d5c80'), btn('Cancelar', '#b91c1c')
    out_q = widgets.Output()

    def on_bus(_):
        with out_q:
            clear_output()
            if not q_cod.value.strip(): alert('Ingrese el codigo de la cita', 'err'); return
            c = svc_cit.buscar(q_cod.value.strip())
            if c:
                p, m = svc_pac.buscar(c.num_paciente), svc_med.buscar(c.num_medico)
                display(HTML(f'<table class="sgh-table"><tr><th>Campo</th><th>Valor</th></tr><tr><td>Codigo</td><td>{c.codigo}</td></tr><tr><td>Paciente</td><td>{p.nombre if p else c.num_paciente}</td></tr><tr><td>Medico</td><td>{m.nombre if m else c.num_medico}</td></tr><tr><td>Fecha/Hora</td><td>{c.fecha} - {c.hora}</td></tr><tr><td>Estado</td><td>{badge(c.estado)}</td></tr></table>'))
            else: alert('Cita no encontrada', 'err')

    def on_canc(_):
        with out_q:
            clear_output()
            if not q_cod.value.strip(): alert('Ingrese el codigo', 'err'); return
            ok, res = svc_cit.cancelar(q_cod.value.strip())
            alert(res if ok else f'Error: {res}', 'ok' if ok else 'err')

    b_bus.on_click(on_bus)
    b_canc.on_click(on_canc)
        # Funcion para mostrar todas las citas
    def mostrar_todas_citas():
        with out_q:
            clear_output()
            citas = svc_cit.listar_todas()
            if citas:
                filas = ''
                for c in citas:
                    p = svc_pac.buscar(c.num_paciente)
                    m = svc_med.buscar(c.num_medico)
                    nombre_pac = p.nombre if p else c.num_paciente
                    nombre_med = m.nombre if m else c.num_medico
                    filas += f'<tr><td>{c.codigo}</td><td>{nombre_pac}</td><td>{nombre_med}</td><td>{c.fecha}</td><td>{c.hora}</td><td>{badge(c.estado)}</td></tr>'
                display(HTML('<table class=\'sgh-table\'><thead><tr><th>Codigo</th><th>Paciente</th><th>Medico</th><th>Fecha</th><th>Hora</th><th>Estado</th></tr></thead><tbody>' + filas + '</tbody></table>'))
            else:
                display(HTML('<p style="color:#888">No hay citas programadas</p>'))

    # Asignar evento al boton Mostrar Todas
    b_list_all.on_click(lambda _: mostrar_todas_citas())

    # Diseño de la pestaña
    tab1 = widgets.VBox([q_cod, widgets.HBox([b_bus, b_list_all, b_canc]), out_q], layout=widgets.Layout(padding='12px'))
    tabs.children = [tab0, tab1]
    for i, t in enumerate(['Programar','Consultar/Cancelar']): tabs.set_title(i, t)
    display(tabs)

def crear_boton_activo(nombre, color):
    return widgets.Button(description=nombre, layout=widgets.Layout(width='auto', height='34px', margin='5px'), style=widgets.ButtonStyle(button_color=color, font_weight='600'))

def main():
    show_css()
    display(HTML('<div class="sgh-header"><span style="font-size:2rem">🏥</span><div><h1>Sistema de Gestion Hospitalaria</h1><p>Programacion Orientada a Objetos</p></div></div>'))
    n_pac = len(svc_pac.listar_todos())
    display(HTML(f'<div class="sgh-card"><div class="sgh-stat-grid"><div class="sgh-stat"><div class="num">{n_pac}</div><div class="lbl">Pacientes</div></div></div></div>'))


btn_pac = crear_boton_activo('Pacientes', '#1a7f4b')
btn_med = crear_boton_activo('Medicos', '#3d5c80')
btn_cit = crear_boton_activo('Citas', '#3d5c80')
nav = widgets.HBox([btn_pac, btn_med, btn_cit], layout=widgets.Layout(gap='0px', padding='12px', background='#1a1a2e', border='1px solid #3d3d5c', flex_wrap='wrap'))
out = widgets.Output()

def reset_botones():
    btn_pac.style.button_color = '#3d5c80'
    btn_med.style.button_color = '#3d5c80'
    btn_cit.style.button_color = '#3d5c80'

def activar_boton(btn):
    reset_botones()
    btn.style.button_color = '#1a7f4b'

def on_pac(_):
    out.clear_output()
    activar_boton(btn_pac)
    panel_pacientes(svc_pac)
def on_med(_):
    out.clear_output()
    activar_boton(btn_med)
    panel_medicos(svc_med, svc_esp)
def on_cit(_):
    out.clear_output()
    activar_boton(btn_cit)
    panel_citas(svc_cit, svc_pac, svc_med)
btn_pac.on_click(on_pac)
btn_med.on_click(on_med)
btn_cit.on_click(on_cit)
display(nav)
display(out)

# === INICIALIZAR ===
_pac_repo = Repositorio(RUTAS['pacientes'], Paciente)
_med_repo = Repositorio(RUTAS['medicos'], Medico)
_esp_repo = Repositorio(RUTAS['especialidades'], Especialidad)
_cit_repo = Repositorio(RUTAS['citas'], Cita)
_con_repo = Repositorio(RUTAS['consultas'], ConsultaMedica)
_tra_repo = Repositorio(RUTAS['tratamientos'], Tratamiento)
_medic_repo = Repositorio(RUTAS['medicamentos'], Medicamento)

svc_pac = PacienteService(_pac_repo)
svc_med = MedicoService(_med_repo)
svc_esp = EspecialidadService(_esp_repo)
svc_cit = CitaService(_cit_repo, _pac_repo, _med_repo)

print('Sistema listo!')
main()


Output()

Sistema listo!
